In [138]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


In [139]:
train = pd.read_csv("ka3_dataset/train.csv")
test = pd.read_csv("ka3_dataset/test.csv")
sample_submission = pd.read_csv("ka3_dataset/sample_submission.csv")

train["is_train"] = 1
test["is_train"] = 0

In [140]:
df = pd.concat([train, test], axis=0)
df.head()

,id,phrase,feature_1,feature_2,feature_3,sentiment,is_train
0,0,It may as well be called `` Jar-Jar Binks : The Movie . '',14.0,5.0,7.0,0.0,1
1,1,You have to see it .,6.0,1.0,NaN,2.0,1
2,2,... either you 're willing to go with this claustrophobic concept or you 're not .,16.0,0.0,6.0,1.0,1
3,3,"Watching Harris ham it up while physically and emotionally disintegrating over the course of the movie has a certain poignancy in light of his recent death , but Boyd 's film offers little else of consequence .",37.0,NaN,3.0,1.0,1
4,4,"Pete 's screenplay manages to find that real natural , even-flowing tone that few movies are able to accomplish .",20.0,1.0,4.0,2.0,1


In [141]:
df.shape, train.shape, test.shape

((8700, 7), (7000, 7), (1700, 6))

### Let's perfrom eda and fe

In [142]:
df.info()

<class 'pandas.DataFrame'>
Index: 8700 entries, 0 to 1699
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         8700 non-null   int64  
 1   phrase     8700 non-null   str    
 2   feature_1  7566 non-null   float64
 3   feature_2  7278 non-null   float64
 4   feature_3  7548 non-null   float64
 5   sentiment  7000 non-null   float64
 6   is_train   8700 non-null   int64  
dtypes: float64(4), int64(2), str(1)
memory usage: 543.8 KB


In [143]:
df.describe()

,id,feature_1,feature_2,feature_3,sentiment,is_train
count,8700.000000,7566.000000,7278.000000,7548.000000,7000.000000,8700.000000
mean,2981.683908,19.026302,1.994367,3.358771,1.041143,0.804598
std,2106.439903,9.283447,1.626308,2.367978,0.898010,0.396533
min,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,1087.000000,12.000000,1.000000,2.000000,0.000000,1.000000
50%,2649.500000,18.000000,1.000000,3.000000,1.000000,1.000000
75%,4824.250000,25.000000,3.000000,4.000000,2.000000,1.000000
max,6999.000000,52.000000,19.000000,23.000000,2.000000,1.000000


In [144]:
# i want to view entire phrase
pd.set_option("display.max_colwidth", None)

df["phrase"].head(5)

0                                                                                                                                                            It may as well be called `` Jar-Jar Binks : The Movie . ''
1                                                                                                                                                                                                  You have to see it .
2                                                                                                                                    ... either you 're willing to go with this claustrophobic concept or you 're not .
3    Watching Harris ham it up while physically and emotionally disintegrating over the course of the movie has a certain poignancy in light of his recent death , but Boyd 's film offers little else of consequence .
4                                                                                                     Pete 's screenplay manages to find

### Handling Phrase feature

In [149]:
df["phrase"]

0                                                                                                                                                               It may as well be called `` Jar-Jar Binks : The Movie . ''
1                                                                                                                                                                                                     You have to see it .
2                                                                                                                                       ... either you 're willing to go with this claustrophobic concept or you 're not .
3       Watching Harris ham it up while physically and emotionally disintegrating over the course of the movie has a certain poignancy in light of his recent death , but Boyd 's film offers little else of consequence .
4                                                                                                        Pete 's screenplay 

In [ ]:
import re
def cleanText(text):
    text = re.sub(r"`", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\.{2,}", " ", text)
    text = text.replace('"', "") 
    return " ".join(text.split())

In [161]:
df["phrase"]= df["phrase"].fillna("")
df["phrase"] = df["phrase"].apply(lambda x: cleanText(x))

In [162]:
df["char_len"] = df["phrase"].str.len()
df["word_len"] = df["phrase"].str.split().str.len()

In [163]:
df["num_!"] = df["phrase"].str.count("!")
df["num_?"] = df["phrase"].str.count(r"\?")

In [165]:
df.head()
# df.groupby("sentiment")[["char_len", "word_len"]].value_counts()

,id,phrase,feature_1,feature_2,feature_3,sentiment,is_train,char_len,word_len,num_!,num_?
0,0,It may as well be called Jar-Jar Binks : The Movie . '',14.0,5.0,7.0,0.0,1,55,13,0,0
1,1,You have to see it .,6.0,1.0,NaN,2.0,1,20,6,0,0
2,2,either you 're willing to go with this claustrophobic concept or you 're not .,16.0,0.0,6.0,1.0,1,78,15,0,0
3,3,"Watching Harris ham it up while physically and emotionally disintegrating over the course of the movie has a certain poignancy in light of his recent death , but Boyd 's film offers little else of consequence .",37.0,NaN,3.0,1.0,1,210,37,0,0
4,4,"Pete 's screenplay manages to find that real natural , even-flowing tone that few movies are able to accomplish .",20.0,1.0,4.0,2.0,1,113,20,0,0


### Handling missing values

In [ ]:
df.isna().sum()

id              0
phrase          0
feature_1    1134
feature_2    1422
feature_3    1152
sentiment    1700
is_train        0
dtype: int64

In [ ]:
df.nunique()

id           7000
phrase       8523
feature_1      52
feature_2      16
feature_3      21
sentiment       3
is_train        2
dtype: int64

In [176]:
num_cols = ["feature_1", "feature_2", "feature_3"]
df[num_cols]

,feature_1,feature_2,feature_3
0,14.0,5.0,7.0
1,6.0,1.0,NaN
2,16.0,0.0,6.0
3,37.0,NaN,3.0
4,20.0,1.0,4.0
...,...,...,...
1695,11.0,2.0,NaN
1696,NaN,2.0,3.0
1697,25.0,1.0,1.0
1698,NaN,1.0,1.0


In [182]:
train_df = df[df["is_train"]==1]
test_df = df[df["is_train"]==0]
train_df.shape, train.shape, test_df.shape, test.shape, 

((7000, 11), (7000, 7), (1700, 11), (1700, 6))

In [171]:
from sklearn.model_selection import train_test_split
X = train_df.drop("sentiment", axis = 1)
y = train_df["sentiment"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify= y
) 

In [188]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, FeatureUnion #used for two or more text vectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

numerical_cols = Pipeline([
    ("impute",SimpleImputer(strategy='median', add_indicator=True)),
    ("scale", StandardScaler(with_mean=False)) # with_mean=False because of sparse matrix from tfidf vectorizer
])

tfidf_pipe = FeatureUnion([
    ("word",TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=30000)),
    ("char", TfidfVectorizer(ngram_range=(3,5), min_df=1, max_features=30000, analyzer="char"))
])

process = ColumnTransformer(transformers=[
    ("nums", numerical_cols, num_cols),
    ("text", tfidf_pipe, "phrase")
])



In [ ]:
X_train_final = process.fit_transform(X_train)
X_val_final = process.transform(X_val)
test_final = process.transform(test_df)

In [194]:
X_train_final.shape, X_val_final.shape, test_final.shape 

((5600, 60006), (1400, 60006), (1700, 60006))

### Model training and evaluation

In [198]:
from sklearn.linear_model import RidgeClassifier, SGDClassifier, LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC

In [199]:
model = {
    "LogisticReg": LogisticRegression(max_iter=1000),
    "RidgeClassifier": RidgeClassifier(),
    "SGDClassifier": SGDClassifier(loss= "log_loss"),
    "LinearSVC": LinearSVC(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "AdaBoostClassifier": AdaBoostClassifier(),
    "GradientBoostingClassifier": GradientBoostingClassifier(),
    "RandomForestClassifier": RandomForestClassifier()
}

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

for name, model in model.items():
    model.fit(X_train_final, y_train)
    y_preds = model.predict(X_val_final)
    print(f"{name} accurace is : {accuracy_score(y_val, y_preds):.4f} \n {classification_report(y_val, y_preds)}")
    print()
